# Pipeline Sanity Check (run on Colab — local machine can't reliably run TensorFlow right now)

Injects a synthetic feature with a KNOWN, planted correlation to future returns, and checks whether the walk-forward LSTM pipeline actually detects it. This does NOT tell us whether real news sentiment predicts real returns — it answers a narrower question: was our earlier ~50% directional accuracy on real data because there's genuinely no signal, or because something in the pipeline can't detect signal even when it's there?

**How to use:** Runtime menu -> Run all. Read the printed RESULTS section at the end.

In [ ]:
!pip install -q yfinance --quiet

In [ ]:
import math

import numpy as np
import pandas as pd
import yfinance as yf
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, r2_score
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.layers import Dense, Dropout, LSTM
from tensorflow.keras.models import Sequential

TICKER = "GOOGL"
YEARS_OF_HISTORY = 5
EPOCHS = 100
TIME_STEP = 100
HORIZON = 3
SIGNAL_STRENGTH = 0.8   # how much of the true future return leaks into the synthetic feature
NOISE_LEVEL = 0.5       # noise added on top, so it's not a perfect giveaway

BASE_FEATURE_COLUMNS = ["Return", "Price_SMA10", "Price_SMA50", "RSI_14", "MACD_norm", "MACD_signal_norm", "Volume_z"]
TARGET_COLUMN = "Target_Return"

In [ ]:
def add_technical_indicators(df: pd.DataFrame, horizon: int) -> pd.DataFrame:
    out = df.copy()
    sma_10 = out["Close"].rolling(10).mean()
    sma_50 = out["Close"].rolling(50).mean()

    out["Return"] = out["Close"].pct_change()
    out["Price_SMA10"] = out["Close"] / sma_10 - 1
    out["Price_SMA50"] = out["Close"] / sma_50 - 1

    delta = out["Close"].diff()
    gain = delta.clip(lower=0).rolling(14).mean()
    loss = (-delta.clip(upper=0)).rolling(14).mean()
    rs = gain / loss.replace(0, np.nan)
    out["RSI_14"] = (100 - (100 / (1 + rs))) / 100

    ema_12 = out["Close"].ewm(span=12, adjust=False).mean()
    ema_26 = out["Close"].ewm(span=26, adjust=False).mean()
    macd = ema_12 - ema_26
    macd_signal = macd.ewm(span=9, adjust=False).mean()
    out["MACD_norm"] = macd / out["Close"]
    out["MACD_signal_norm"] = macd_signal / out["Close"]

    vol_mean = out["Volume"].rolling(20).mean()
    vol_std = out["Volume"].rolling(20).std()
    out["Volume_z"] = (out["Volume"] - vol_mean) / vol_std.replace(0, np.nan)

    out["Target_Return"] = out["Close"].shift(-horizon) / out["Close"] - 1
    return out.dropna().reset_index(drop=True)


def create_dataset(feature_arr: np.ndarray, target_arr: np.ndarray, time_step: int):
    X, Y = [], []
    for i in range(time_step - 1, len(feature_arr)):
        X.append(feature_arr[i - time_step + 1: i + 1, :])
        Y.append(target_arr[i])
    return np.array(X), np.array(Y)


def build_model(time_step: int, num_features: int) -> Sequential:
    # Simplified vs the main app's 3-stacked-LSTM+dropout architecture: that combo
    # was too heavy for ~500-600 training samples and tiny return-scale targets,
    # and was collapsing to predicting close to the mean regardless of input
    # (confirmed by control and signal-injected runs producing identical accuracy).
    model = Sequential([
        LSTM(32, input_shape=(time_step, num_features)),
        Dense(16, activation="relu"),
        Dense(1),
    ])
    model.compile(loss="mean_squared_error", optimizer="adam")
    return model


def run_fold(train_df, test_df, feature_columns, epochs, time_step):
    feature_scaler = MinMaxScaler(feature_range=(0, 1))
    train_feat_scaled = feature_scaler.fit_transform(train_df[feature_columns].values)
    test_feat_scaled = feature_scaler.transform(test_df[feature_columns].values)

    target_scaler = MinMaxScaler(feature_range=(0, 1))
    train_target_scaled = target_scaler.fit_transform(train_df[[TARGET_COLUMN]].values).flatten()
    test_target_scaled = target_scaler.transform(test_df[[TARGET_COLUMN]].values).flatten()

    X_train, Y_train = create_dataset(train_feat_scaled, train_target_scaled, time_step)
    X_test, Y_test = create_dataset(test_feat_scaled, test_target_scaled, time_step)
    if X_train.size == 0 or X_test.size == 0:
        return None

    model = build_model(time_step, len(feature_columns))
    early_stop = EarlyStopping(monitor="val_loss", patience=10, restore_best_weights=True)
    history = model.fit(X_train, Y_train, validation_split=0.1, epochs=epochs, batch_size=64,
                         callbacks=[early_stop], verbose=0)

    pred_return_scaled = model.predict(X_test, verbose=0).flatten()
    pred_return = target_scaler.inverse_transform(pred_return_scaled.reshape(-1, 1)).flatten()
    actual_return = test_df[TARGET_COLUMN].values[time_step - 1:]

    base_close = test_df["Close"].values[time_step - 1:]
    predicted_price = base_close * (1 + pred_return)
    actual_price = base_close * (1 + actual_return)
    naive_price = base_close

    # Diagnostic: if pred_std is much smaller than actual_std, the model has
    # collapsed to predicting close to a constant (the classic "cheap MSE floor").
    print(f"    epochs run: {len(history.history['loss'])}, "
          f"pred_return std: {pred_return.std():.5f}, actual_return std: {actual_return.std():.5f}")

    return {
        "test_rmse": math.sqrt(mean_squared_error(actual_price, predicted_price)),
        "naive_rmse": math.sqrt(mean_squared_error(actual_price, naive_price)),
        "r2": r2_score(actual_price, predicted_price),
        "dir_acc": float(np.mean(np.sign(pred_return) == np.sign(actual_return)) * 100),
    }


def single_split_eval(featured: pd.DataFrame, feature_columns: list, label: str):
    train_size = int(len(featured) * 0.8)
    train_df = featured.iloc[:train_size].reset_index(drop=True)
    test_df = featured.iloc[train_size:].reset_index(drop=True)
    if len(train_df) <= TIME_STEP + 1 or len(test_df) <= TIME_STEP + 1:
        print(f"  [{label}] not enough data for this split")
        return None

    print(f"  [{label}] training...", flush=True)
    result = run_fold(train_df, test_df, feature_columns, EPOCHS, TIME_STEP)
    if result is None:
        print(f"  [{label}] no usable data after windowing")
        return None

    return {
        "mean_test_rmse": result["test_rmse"],
        "mean_naive_rmse": result["naive_rmse"],
        "mean_r2": result["r2"],
        "mean_dir_acc": result["dir_acc"],
    }

In [ ]:
print(f"Downloading {TICKER}...")
raw = yf.download(TICKER, period=f"{YEARS_OF_HISTORY}y")
if isinstance(raw.columns, pd.MultiIndex):
    raw.columns = raw.columns.get_level_values(0)

featured = add_technical_indicators(raw[["Close", "Volume"]], HORIZON)

print("\n=== CONTROL: real technical features only (no synthetic signal) ===")
control = single_split_eval(featured, BASE_FEATURE_COLUMNS, "control")

rng = np.random.default_rng(42)
noise = rng.normal(0, featured[TARGET_COLUMN].std() * NOISE_LEVEL, size=len(featured))
featured["Synthetic_Signal"] = featured[TARGET_COLUMN] * SIGNAL_STRENGTH + noise
signal_columns = BASE_FEATURE_COLUMNS + ["Synthetic_Signal"]

print("\n=== SIGNAL: same features + a planted, partially-correlated synthetic feature ===")
signal = single_split_eval(featured, signal_columns, "signal")

In [ ]:
print("=" * 60)
print("RESULTS")
print("=" * 60)
for label, res in [("Control (real features only)", control), ("Signal-injected", signal)]:
    if res is None:
        print(f"{label}: FAILED")
        continue
    print(f"{label}:")
    print(f"  Mean test RMSE:        {res['mean_test_rmse']:.3f}  (naive baseline: {res['mean_naive_rmse']:.3f})")
    print(f"  Mean R2:               {res['mean_r2']:.3f}")
    print(f"  Mean directional acc:  {res['mean_dir_acc']:.1f}%")
    print()

if control and signal:
    if signal["mean_dir_acc"] > control["mean_dir_acc"] + 15 and signal["mean_test_rmse"] < signal["mean_naive_rmse"]:
        print("PIPELINE CHECK PASSED: the model clearly detected the planted signal.")
        print("Our earlier real-data findings (~50% directional accuracy) are trustworthy —")
        print("the pipeline CAN detect real signal when it's there; it just didn't find any in the real data.")
    else:
        print("PIPELINE CHECK FAILED (or inconclusive): the model did NOT clearly pick up the planted signal.")
        print("This suggests something in the pipeline may be preventing real signal from being learned —")
        print("worth investigating before trusting the earlier 'no signal found' conclusions.")